In [0]:
# Install required packages
%pip install transformers sacremoses sentencepiece datasets
%pip install --upgrade pip
%pip install torch


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.7/796.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 73.5 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2024.11.6
    Not uninstalling regex at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-9a34b5bc-a57a-455d-b9d4-57327828b132
    Can't uninstall 'regex'. No files were found to uninstall.
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.2.0
    Not unins

In [0]:
from transformers import MarianMTModel, MarianTokenizer

class ModernTranslator:
    def __init__(self):
        self.models = {}
        self.tokenizers = {}
    
    def load_model(self, source_lang, target_lang):
        model_name = f'Helsinki-NLP/opus-mt-{source_lang}-{target_lang}'
        if model_name not in self.models:
            self.models[model_name] = MarianMTModel.from_pretrained(model_name)
            self.tokenizers[model_name] = MarianTokenizer.from_pretrained(model_name)
        return self.models[model_name], self.tokenizers[model_name]
    
    def translate(self, text, source_lang='en', target_lang='es'):
        model, tokenizer = self.load_model(source_lang, target_lang)
        
        # Tokenize and translate
        inputs = tokenizer(text, return_tensors="pt", padding=True)
        translated = model.generate(**inputs)
        
        # Decode the translation
        result = tokenizer.decode(translated[0], skip_special_tokens=True)
        return result


/local_disk0/.ephemeral_nfs/envs/pythonEnv-9a34b5bc-a57a-455d-b9d4-57327828b132/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
class BatchTranslator(ModernTranslator):
    def batch_translate(self, texts, source_lang='en', target_lang='es', batch_size=32):
        model, tokenizer = self.load_model(source_lang, target_lang)
        results = []
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True)
            translated = model.generate(**inputs)
            decoded = [tokenizer.decode(t, skip_special_tokens=True) 
                      for t in translated]
            results.extend(decoded)
        
        return results


In [0]:
from transformers import pipeline

class AdvancedTranslator(BatchTranslator):
    def __init__(self):
        super().__init__()
        self.qa_pipeline = pipeline("question-answering")
    
    def translate_with_context(self, text, context, source_lang='en', target_lang='es'):
        """Translates text while maintaining context awareness"""
        # Combine text with context
        combined_text = f"Context: {context}\nText: {text}"
        translation = self.translate(combined_text, source_lang, target_lang)
        
        # Extract relevant part of translation
        return self._extract_translation(translation)
    
    def _extract_translation(self, full_translation):
        # Use QA pipeline to extract the relevant translated part
        question = "What is the translated text?"
        result = self.qa_pipeline(question=question, context=full_translation)
        return result['answer']


In [0]:
def main():
    # Initialize translator
    translator = BatchTranslator()
    
    # Single translation
    text = "Hello, how are you?"
    translated = translator.translate(text, 'en', 'es')
    print(f"Original: {text}")
    print(f"Translated: {translated}")
    
    # Batch translation
    texts = [
        "Hello, how are you?",
        "I am learning NLP",
        "Machine translation is fascinating"
    ]
    translations = translator.batch_translate(texts, 'en', 'es')
    for original, translated in zip(texts, translations):
        print(f"\nOriginal: {original}")
        print(f"Translated: {translated}")
    


if __name__ == "__main__":
    main()


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  312MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

... (2 outputs left out)

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Original: Hello, how are you?
Translated: Hola, ¿cómo estás?

Original: Hello, how are you?
Translated: Hola, ¿cómo estás?

Original: I am learning NLP
Translated: Estoy aprendiendo NLP

Original: Machine translation is fascinating
Translated: Traducción automática es fascinante
